<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session04-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 4: in-class discussion problem (2 of 3)

This is the block 2 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then run the code below and look at what it does. There is one mistake in it, but finding that is
only the way in: the point is to work out why the result comes out as it does. Discuss that with
your neighbour, then answer the quiz again.

## Why does affine care about scattering, but linear doesn't?

A gap of four residues can sit in an alignment as one block, or as four single gaps scattered along
the row. A linear gap penalty charges $g$ per gap position. An affine penalty charges
$g_{\text{open}}$ for the first position of each gap and $g_{\text{ext}}$ for each further one:
$g(n) = g_{\text{open}} + (n - 1)\,g_{\text{ext}}$. With linear $g = -1$ and affine
$g_{\text{open}} = -8$, $g_{\text{ext}} = -1$, what does each scheme charge for the block, and for the
scattered gaps?

**Fix and run.** The function below reads an aligned row character by character and adds up its gap
penalty. It forgets something, at the place marked `FIX`. Run the cell once as it is: does affine
prefer the block? Then fix the function and run it again.

In [1]:
def gap_cost(row, g_open, g_ext):
    """The gap penalty of one aligned row: g_open for the first '-' of a gap, g_ext for each further one."""
    cost, in_gap = 0, False
    for ch in row:
        if ch == "-":
            cost += g_ext if in_gap else g_open
            in_gap = True
        # FIX: a residue ends the gap, but nothing here says so
    return cost

block     = "MV----HLTPEEK"
scattered = "M-VH-LT-PE-EK"
for name, g_open, g_ext in [("linear", -1, -1), ("affine", -8, -1)]:
    print(f"{name:7s} block {gap_cost(block, g_open, g_ext):4d}   scattered {gap_cost(scattered, g_open, g_ext):4d}")

linear  block   -4   scattered   -4
affine  block  -11   scattered  -11


**If you finish early.**

1. How large must $g_{\text{open}}$ be, with $g_{\text{ext}} = -1$, for affine to charge the scattered
   gaps at least twice as much as the block?
2. Real insertions and deletions usually remove or add several residues at once. Which scheme fits
   that biology better?

<details>
<summary><b>Discussion point</b> &mdash; open this once you have run the code and talked it through</summary>

The function never noticed that a residue ends a gap: once `in_gap` was
`True`, it stayed `True`, so every later gap was charged as an extension of the first one. The fix is
two lines, `else: in_gap = False`. As it was, affine charged $-11$ for both rows and looked just as
indifferent as linear.

Fixed, linear charges **−4 either way**: its cost depends only on the total gap length. Affine charges
**−11 for the block** ($-8 - 3$) but **−32 for the scattered gaps** ($4 \times -8$), because each
isolated gap pays its own opening cost. That is why affine penalties keep insertions and deletions
together in blocks, the way they happen in evolution. For the first question: the block costs
$g_{\text{open}} - 3$ and the scattered gaps $4\,g_{\text{open}}$, so twice as much needs
$4\,g_{\text{open}} \le 2(g_{\text{open}} - 3)$, that is $g_{\text{open}} \le -3$.

</details>